<a href="https://colab.research.google.com/github/jbalcazarusistemas-prog/Intep-ciencia-de-datos/blob/jefferson-drone/DivisionsRealTime_(2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import requests
import sys
import json
import base64
import pyspark.pandas as pd
from pyspark.sql.functions import lit, col,explode
from pyspark.sql.types import StructType, StructField, StringType, DecimalType, DoubleType, MapType, IntegerType, ArrayType
import os

/usr/local/lib/python3.12/dist-packages/pyspark/pandas/__init__.py:43: UserWarning: 'PYARROW_IGNORE_TIMEZONE' environment variable was not set. It is required to set this environment variable to '1' in both driver and executor sides if you use pyarrow>=2.0.0. pandas-on-Spark will set it for you but it does not work if there is a Spark context already launched.
  warnings.warn(


In [4]:
!pip install --upgrade pyspark

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 434.2/434.2 MB 1.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.0/203.0 kB 4.0 MB/s eta 0:00:00
  Created wheel for pyspark: filename=pyspark-4.0.1-py2.py3-none-any.whl size=434813800 sha256=413c1044723f4341b61477017fe5268acc880d878bd88e15c9c08a639dea3749
  Stored in directory: /root/.cache/pip/wheels/31/9f/68/f89fb34ccd886909be7d0e390eaaf97f21efdf540c0ee8dbcd
Successfully built pyspark
  Attempting uninstall: py4j
    Found existing installation: py4j 0.10.9.7
    Uninstalling py4j-0.10.9.7:
      Successfully uninstalled py4j-0.10.9.7
  Attempting uninstall: pyspark
    Found existing installation: pyspark 3.5.1
    Uninstalling pyspark-3.5.1:
      Successfully uninstalled pyspark-3.5.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-conne

In [2]:
from pyspark.sql import SparkSession

# Initialize SparkSession if it's not already initialized
if 'spark' not in locals() or spark is None:
    spark = SparkSession.builder.appName("StreamProcessor").getOrCreate()

df_rate = spark.readStream.format("rate").option("rowsPerSecond", 10).load() ## Crear datos simulados.

In [3]:

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, struct, to_json, window

# 1. Fuente simulada: genera 10 filas por segundo
df_rate = spark.readStream.format("rate").option("rowsPerSecond", 10).load()
# 2. Enriquecer con columnas adicionales
df_enriched = df_rate.withColumn("category", lit("Simulado")) \
                     .withColumn("event_type", lit("click")) \
                     .withColumn("random_score", (col("value") % 100)) \
                     .withColumn("description", lit("Evento generado en tiempo real"))
# 3. Convertir a JSON (opcional, para enviar a sistemas externos)
df_json = df_enriched.withColumn(
    "payload",
    to_json(
        struct(
            col("timestamp").alias("event_time"),
            col("value").alias("event_id"),
            col("category"),
            col("event_type"),
            col("random_score"),
            col("description")
        )
    )
)

# 4. Agregación por ventana (conteo cada 10 segundos)
df_agg = df_enriched.groupBy(
    window(col("timestamp"), "10 seconds")
).count()

df_final = df_agg.select(
    col("window.start").alias("start_time"),
    col("window.end").alias("end_time"),
    col("count").alias("event_count")
)

# 5. Visualización en tiempo real en Databricks

display(df_final)



DataFrame[start_time: timestamp, end_time: timestamp, event_count: bigint]

#Ejercicios:


##Ejercicio 1: Lecturas de temperatura de sensores IoT
- Problema real: Monitoreo de temperatura en diferentes puntos de una planta industrial.
- Simulación: 3 sensores registrando temperatura cada hora durante 24 horas.
- Gráfica: Línea mostrando la evolución de la temperatura por sensor.



## Generar Datos de Temperatura Simulados


Generar un DataFrame de PySpark con datos simulados de temperatura para 3 sensores, registrando lecturas cada hora durante 24 horas. El DataFrame incluirá columnas para 'timestamp', 'sensor_id' y 'temperature'.


In [4]:
from datetime import datetime, timedelta
import random
from pyspark.sql.types import StructType, StructField, TimestampType, StringType, DoubleType

# 1. Hora de inicio de la simulacion
start_time = datetime(2023, 1, 1, 0, 0, 0)

# 2. Cree una lista de marcas de tiempo (24 horas, una entrada por hora)
timestamps = [start_time + timedelta(hours=i) for i in range(24)]

# 3. Inicialice una lista vacía para almacenar todos los datos
all_sensor_data = []

# 4. Genere lecturas de temperatura para 3 sensores.
for sensor_id_num in range(1, 4):
    sensor_name = f"sensor_{sensor_id_num}"
    for ts in timestamps:
        # Genere temperatura aleatoria entre 15,0 y 30,0
        temperature = round(random.uniform(15.0, 30.0), 2)
        all_sensor_data.append({
            "timestamp": ts,
            "sensor_id": sensor_name,
            "temperature": temperature
        })

print(f"Genero {len(all_sensor_data)} puntos de datos para simulación.")

Genero 72 puntos de datos para simulación.


Ahora que los datos simulados están preparados como una lista de diccionarios, el siguiente paso es definir el esquema para PySpark DataFrame y luego crear el DataFrame mismo usando `spark.createDataFrame()`. Finalmente, se mostrarán las primeras filas del DataFrame y su esquema para confirmar la generación de datos y la aplicación del esquema exitosas.


In [5]:
from pyspark.sql.types import StructType, StructField, TimestampType, StringType, DoubleType

# 5. Definir esquema para el DataFrame
temperature_schema = StructType([
    StructField("timestamp", TimestampType(), True),
    StructField("sensor_id", StringType(), True),
    StructField("temperature", DoubleType(), True)
])

# 6. Crear PySpark DataFrame
df_temperatures = spark.createDataFrame(all_sensor_data, schema=temperature_schema)

# 7. Mostrar las primeras filas y el esquema.
print("Mostrando las primeras 5 filas de df_temperatures:")
df_temperatures.show(5)
print("\nMostrando el esquema de df_temperatures:")
df_temperatures.printSchema()

Mostrando las primeras 5 filas de df_temperatures:
+-------------------+---------+-----------+
|          timestamp|sensor_id|temperature|
+-------------------+---------+-----------+
|2023-01-01 00:00:00| sensor_1|       15.1|
|2023-01-01 01:00:00| sensor_1|      18.19|
|2023-01-01 02:00:00| sensor_1|       19.2|
|2023-01-01 03:00:00| sensor_1|      24.67|
|2023-01-01 04:00:00| sensor_1|      18.44|
+-------------------+---------+-----------+
only showing top 5 rows

Mostrando el esquema de df_temperatures:
root
 |-- timestamp: timestamp (nullable = true)
 |-- sensor_id: string (nullable = true)
 |-- temperature: double (nullable = true)



## Visualizar Evolución de Temperatura

Crear una gráfica de líneas mostrando la evolución de la temperatura a lo largo del tiempo para cada uno de los 3 sensores. Se asegurará que la gráfica sea clara, con leyendas y etiquetas adecuadas para los ejes y un título descriptivo.


El primer paso es convertir PySpark DataFrame `df_temperatures` en un Pandas DataFrame, ya que Plotly Express funciona directamente con Pandas DataFrames. Luego, importar `plotly.express` para trazar y procederé a generar el gráfico de líneas según las instrucciones.



In [6]:
import plotly.express as px

# 1. Convertir PySpark DataFrame a Pandas DataFrame
pd_df_temperatures = df_temperatures.toPandas()

# 2. Crear una linea plot usando plotly.express.line
fig = px.line(
    pd_df_temperatures,
    x="timestamp",
    y="temperature",
    color="sensor_id",
    title="Temperature Evolution of IoT Sensors Over 24 Hours",
    labels={
        "timestamp": "Time",
        "temperature": "Temperature (°C)",
        "sensor_id": "Sensor ID"
    }
)

# Mostrar la trama
fig.show()


##Ejercicio 2: Ventas por categoría de producto
- Problema real: Análisis de ventas por categoría en un e-commerce.
- Simulación: Total de ventas por 5 categorías.
- Gráfica: Barras mostrando ventas totales por categoría.


## Generar Datos de Ventas Simulados

### Subtask:
Generar un DataFrame de PySpark con datos simulados de ventas para 5 categorías, incluyendo 'category', 'product_id', 'price' y 'quantity'.


Importar los módulos necesarios, definiré categorías de productos y luego generaré datos de ventas simulados para cada categoría, incluido el ID del producto, el precio y la cantidad, y almacenaré todas las entradas en una lista de diccionarios como se especifica en las instrucciones 1 a 6.



In [7]:
import random
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType

# 1. (datetime no es necesario para esta simulación específica, tipos aleatorios y pyspark importados arriba)

# 2. Definir una lista de 5 categorías de productos.
categories = ["Electronics", "Clothing", "Home Goods", "Books", "Groceries"]

# 3. Inicialice una lista vacía para almacenar todos los datos de ventas simuladas
all_sales_data = []

# 4-6. Recorra cada categoría y genere entradas de ventas
product_id_counter = 1
for category in categories:
    # Genera entre 20 y 50 entradas de ventas por categoría.
    num_sales = random.randint(20, 50)
    for _ in range(num_sales):
        product_id = f"prod_{product_id_counter:04d}"
        price = round(random.uniform(10.00, 500.00), 2)
        quantity = random.randint(1, 10)
        all_sales_data.append({
            "category": category,
            "product_id": product_id,
            "price": price,
            "quantity": quantity
        })
        product_id_counter += 1

print(f"Genero {len(all_sales_data)} puntos de datos de ventas.")

Genero 158 puntos de datos de ventas.


Ahora que los datos de ventas simulados están preparados como una lista de diccionarios, el siguiente paso es definir el esquema para el PySpark DataFrame, crear el propio DataFrame usando `spark.createDataFrame()` y luego mostrar sus primeras filas y el esquema para confirmar la generación exitosa de datos y la aplicación del esquema, según las instrucciones 7-10.



In [8]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType

# 7. Definir esquema para el DataFrame
sales_schema = StructType([
    StructField("category", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("quantity", IntegerType(), True)
])

# 8. Crear PySpark DataFrame
df_sales = spark.createDataFrame(all_sales_data, schema=sales_schema)

# 9. Mostrar las primeras filas
print("Mostrar las primeras 5 filas de df_sales:")
df_sales.show(5)

# 10. imprimir el esquema
print("\nMostrando el esquema de df_sales:")
df_sales.printSchema()

Mostrar las primeras 5 filas de df_sales:
+-----------+----------+------+--------+
|   category|product_id| price|quantity|
+-----------+----------+------+--------+
|Electronics| prod_0001|192.28|       3|
|Electronics| prod_0002|186.29|       2|
|Electronics| prod_0003|195.63|       8|
|Electronics| prod_0004| 95.35|       1|
|Electronics| prod_0005|270.98|      10|
+-----------+----------+------+--------+
only showing top 5 rows

Mostrando el esquema de df_sales:
root
 |-- category: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- price: double (nullable = true)
 |-- quantity: integer (nullable = true)



## Calcular Ventas Totales por Categoría

Calcular las ventas totales por categoría, añadiendo una nueva columna 'total_sales' (price * quantity) y luego agrupando por 'category' para obtener la suma total de ventas para cada una.

1. Agregue una nueva columna llamada "total_sales" a "df_sales" multiplicando las columnas "precio" y "cantidad".
2. Agrupe el DataFrame por "categoría" y calcule la suma de "total_sales" para cada categoría.
3. Muestre el DataFrame resultante que muestra las ventas totales por categoría.

In [9]:
from pyspark.sql.functions import col, sum

# 1. Agregue una nueva columna llamada `total_sales` a `df_sales`
df_sales_with_total = df_sales.withColumn("total_sales", col("price") * col("quantity"))

# 2. Agrupe el DataFrame por "categoría" y calcule la suma de "total_sales" para cada categoría
df_total_sales_per_category = df_sales_with_total.groupBy("category").agg(sum("total_sales").alias("total_sales_amount"))

# 3. Muestre el DataFrame resultante que muestra las ventas totales por categoría.
print("Total ventas por categoria:")
df_total_sales_per_category.show()

Total ventas por categoria:
+-----------+------------------+
|   category|total_sales_amount|
+-----------+------------------+
|Electronics|          47871.32|
|   Clothing|          34582.99|
| Home Goods|          48954.38|
|  Groceries|32722.420000000002|
|      Books| 43948.29000000001|
+-----------+------------------+



## Visualizar Ventas Totales por Categoría

Visualizar las ventas totales por categoría utilizando un gráfico de barras. La gráfica debe ser clara, con etiquetas adecuadas para los ejes y un título descriptivo.

In [10]:
import plotly.express as px

# 1. Convert PySpark DataFrame to Pandas DataFrame for plotting
pd_df_total_sales = df_total_sales_per_category.toPandas()

# 2. Create a bar chart using plotly.express.bar
fig = px.bar(
    pd_df_total_sales,
    x="category",
    y="total_sales_amount",
    title="Total Sales by Product Category",
    labels={
        "category": "Product Category",
        "total_sales_amount": "Total Sales Amount"
    },
    color="category" # Use different colors for each category
)

# 3. Display the plot
fig.show()